# Flower Classification Project

Your task is to **classify photos of flowers as roses, sunflowers or tulips**, with only 100 photos per kind of flower to learn from, using **transfer learning**.

# The Business Problem

A garden centre runs a website where customers upload a photo of a flower from their garden and receive a care guide for it: when to water, when to cut it back, which pests to watch for. At the moment, a member of staff looks at every photo and chooses the right guide by hand. The garden centre starts with its three most requested guides, **roses, sunflowers and tulips**, and wants a model that chooses the guide automatically from the photo.

Your model answers this question: given a photo, which of the three flowers does it show? A wrong answer sends a customer the wrong care guide, so at the end you look at **which photos the model gets wrong**, so that the garden centre knows when to trust it and when a member of staff should check the answer.

The model can only answer with one of the three flowers. A photo of a daisy would still receive one of the three labels, so the website has to make clear which flowers the service covers.

# Data Description

The data ships with this repository in the folder `data/`: **390 colour photos**, one JPEG file per photo, sorted into one folder per kind of flower. The name of the folder is the label of the photo.

| Folder | Photos | Description |
| :-- | :-- | :-- |
| `data/train/rose` | 100 | roses, for training |
| `data/train/sunflower` | 100 | sunflowers, for training |
| `data/train/tulip` | 100 | tulips, for training |
| `data/validation/rose` | 20 | roses, for choosing the model |
| `data/validation/sunflower` | 20 | sunflowers, for choosing the model |
| `data/validation/tulip` | 20 | tulips, for choosing the model |
| `data/test/rose` | 10 | roses, **the test set** |
| `data/test/sunflower` | 10 | sunflowers, **the test set** |
| `data/test/tulip` | 10 | tulips, **the test set** |

The photos have different sizes, from 159 to 640 pixels wide, and show the flowers in very different ways: a single flower close up, a whole field, a bunch in a vase, sometimes with people or buildings in the picture.

The photos are part of the [TensorFlow flower photos](https://www.tensorflow.org/datasets/catalog/tf_flowers) collection and were taken from Flickr. Every photo is licensed under [CC BY 2.0](https://creativecommons.org/licenses/by/2.0/); its photographer is listed in `data/LICENSE.txt`.

## Task

1. **Get the data.** Display a few photos of every class from `data/train`.
2. **Load the training and the validation images.** Load `data/train` and `data/validation` with `keras.utils.image_dataset_from_directory`, with one `image_size` that you use for every model, for example `(160, 160)`. Shuffle the training images with a fixed `seed`; keep the validation images in their order with `shuffle=False`. Do not load `data/test` until step 6.
3. **Explore the training set.** Display a grid of training photos with their labels. What makes the three flowers look different: colour, shape, the background?
4. **Repurpose a pretrained network.** Use `MobileNetV2` from `keras.applications` as the pretrained network and repurpose it to classify the three flowers. Plot its learning curves: does it overfit?

> [!IMPORTANT]
> **Every image has to be prepared in exactly the same way**: the same `image_size`, the same `preprocess_input` of the base, and the same frozen base, for the training, the validation and later the test images.

5. **If needed, improve the model with data augmentation**, and compare the validation accuracy with step 4. Choose your best model by its validation accuracy. **Save every model to a file** with `model.save`, for example `models/dense_head.keras`, so that you can load all of them in step 6 with `keras.models.load_model`, even after restarting the notebook.
6. **Evaluate all models on the test set at the very end.** Load your saved models, and load `data/test` with the same `image_size` and with `shuffle=False`. Then:
    1. Collect the **validation and test accuracy of every model** in **one table**. Is the validation accuracy of your chosen model a fair estimate of its test accuracy? The test set has only 30 photos, so **one photo changes the test accuracy by 3.3 percentage points**.
    2. **Plot the confusion matrix** of your best model on the test set. Which flowers does it confuse?
    3. **Display every misclassified test photo**, with its true and predicted label. What do these photos have in common?
7. **Try your model in an app.** The repository contains a small [Streamlit](https://docs.streamlit.io/) app, `app.py`, that loads a model from `models/` and classifies a photo that you upload. Start it in a terminal in the repository folder with `uv run streamlit run app.py`, choose your best model, and upload a few flower photos of your own. Does the model get them right? The app works with models that take the raw photos and include their preprocessing, such as `preprocess_input`, as layers.

In [ ]:
# Import the libraries you need for this project
from pathlib import Path

import keras
import keras_tuner as kt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from keras.applications import mobilenet_v2
from matplotlib.colors import hsv_to_rgb, rgb_to_hsv, to_hex
from PIL import Image
from plotly.subplots import make_subplots

from functions import (
    build_model,
    colour_name,
    count_correct_and_wrong,
    history_to_frame,
    load_images,
    make_strong_augmentation,
    plot_confusion_and_mistakes,
    plot_learning_curves,
    summarize_curves,
)


## Get the data

Display a few photos of every class from `data/train`

In [ ]:
FLOWERS = Path("data")
IMAGE_SIZE = (224, 224)

> Since the description said the images are differently sized, I wanted to check first how different they were before loading and resizing them with Keras. 

In [ ]:
rows = []
for path in sorted((FLOWERS / "train").glob("*/*.jpg")):
    with Image.open(path) as img:
        width, height = img.size # just read the size and get the width and height of the photos
    rows.append({"class": path.parent.name, "file": path.name, "width": width, "height": height})

sizes = pd.DataFrame(rows)
sizes["aspect_ratio"] = sizes["width"] / sizes["height"] 

print("Photo count:", len(sizes))
print("Different sizes:", sizes[["width", "height"]].drop_duplicates().shape[0])
display(sizes[["width", "height", "aspect_ratio"]].describe().round(2))

# build each plot with plotly express, then put the three side by side in one figure
hist_width = px.histogram(sizes, x="width", nbins=20)
hist_height = px.histogram(sizes, x="height", nbins=20)
scatter = px.scatter(sizes, x="width", y="height", color="class", hover_data=["file", "aspect_ratio"], opacity=0.6)

fig = make_subplots(rows=1, cols=3, subplot_titles=("Width", "Height", "Width vs. Height"))
for trace in hist_width.data:
    fig.add_trace(trace, row=1, col=1)
for trace in hist_height.data:
    fig.add_trace(trace, row=1, col=2)
for trace in scatter.data:
    fig.add_trace(trace, row=1, col=3)

fig.update_xaxes(title_text="Width (px)", row=1, col=1)
fig.update_xaxes(title_text="Height (px)", row=1, col=2)
fig.update_xaxes(title_text="Width (px)", row=1, col=3)
fig.update_yaxes(title_text="Photo Count", row=1, col=1)
fig.update_yaxes(title_text="Height (px)", row=1, col=3)
fig.update_layout(width=1200, height=400, showlegend=True)
fig.show()

In [ ]:
px.box(sizes, 
       y='aspect_ratio',
       color='class',
       title='Aspect ratio of the classes')

> The standard deviation of 116 on the width and 61 on the height is quite big with a mean of 381 width and 286 height. 

> Most of the photos are in landscape format

> Since the width and height is similarily distributed for all classes as well as the aspect ratio is very similarily distributed for all of them, I feel comfortable resizing them all to a specific size since resizing does distort all classes in a similiar fashion. The model won't learn that tulip = strongly compressed or sth like that.
> Most photos are 4:3, so resizing to 224×224 only squeezes them mildly.

## Load the training and the validation images

In [ ]:
train_ds = load_images("train", shuffle=True, data_dir=FLOWERS, image_size=IMAGE_SIZE)
valid_ds = load_images("validation", shuffle=False, data_dir=FLOWERS, image_size=IMAGE_SIZE)
test_ds = load_images("test", shuffle=False, data_dir=FLOWERS, image_size=IMAGE_SIZE)
class_names = train_ds.class_names
print("Classes:", class_names)

## Explore the training set

Display a grid of training photos with their labels. What makes the three flowers look different: colour, shape, the background?

In [ ]:
rng = np.random.default_rng(42)
class_folders = sorted(p for p in (FLOWERS / "train").iterdir() if p.is_dir())

# one row per class, 6 random training photos next to each other
fig, axes = plt.subplots(len(class_folders), 6, figsize=(18, 9))
for row, folder in zip(axes, class_folders):
    photos = sorted(folder.glob("*.jpg"))
    for ax, i in zip(row, rng.choice(len(photos), size=6, replace=False)):
        # load the photo resized to IMAGE_SIZE, as the model will see it
        ax.imshow(keras.utils.load_img(photos[i], target_size=IMAGE_SIZE))
        ax.set_title(folder.name)
        ax.axis("off")
plt.tight_layout()
plt.show()

> Sunflowers have a particular colour, so I would guess the most common colours in the sunflower photo would be yellows, browns and greens. 

> Roses and Tulips can have a lot of different colours but if we are talking about flower bouquet, I guess it is more common to use roses of the same colour together.

> I would like to check which colour values are common for each of the classes and compare. 

In [ ]:
# how many training photos does each class have?
class_counts = sizes["class"].value_counts().sort_index()

px.bar(class_counts, labels={"value": "Photo count", "class": "class"}, title="Training photos per class").show()

In [ ]:
BIN_WIDTH = 5  # degrees of hue per bin
bins = np.arange(0, 360 + BIN_WIDTH, BIN_WIDTH)

hue_rows = []
for folder in class_folders:
    hues = []
    n_pixels = 0
    for path in sorted(folder.glob("*.jpg")):
        rgb = keras.utils.load_img(path, target_size=IMAGE_SIZE)
        hsv = rgb_to_hsv(np.asarray(rgb) / 255).reshape(-1, 3)
        n_pixels += len(hsv)
        # keep only clearly coloured pixels: drop grey (low saturation), dark and washed-out ones
        colourful = hsv[(hsv[:, 1] > 0.3) & (hsv[:, 2] > 0.2) & (hsv[:, 2] < 0.95)]
        hues.append(colourful[:, 0] * 360)
    hues = np.concatenate(hues)
    counts, _ = np.histogram(hues, bins=bins)
    print(f"{folder.name}: {len(hues) / n_pixels:.0%} of the pixels are clearly coloured")
    for left, count in zip(bins[:-1], counts):
        hue_rows.append({"class": folder.name, "hue": left + BIN_WIDTH / 2, "share": 100 * count / len(hues)})

hue_df = pd.DataFrame(hue_rows)

hue_lines = px.line(hue_df, x="hue", y="share", color="class")

# colour strip: one cell per hue bin, painted in the fully saturated colour of that hue
centers = bins[:-1] + BIN_WIDTH / 2
strip_colors = [to_hex(hsv_to_rgb((h / 360, 1, 1))) for h in centers]
strip = go.Heatmap(
    x=centers,
    z=[centers],
    zmin=0,
    zmax=360,
    colorscale=[[i / (len(centers) - 1), c] for i, c in enumerate(strip_colors)],
    showscale=False,
    hovertemplate="Hue: %{x}°<extra></extra>",
)

# lines on top, colour strip below, sharing the hue axis
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.88, 0.12], vertical_spacing=0.03)
for trace in hue_lines.data:
    fig.add_trace(trace, row=1, col=1)
fig.add_trace(strip, row=2, col=1)

# name the main colours on the hue axis
fig.update_xaxes(
    range=[0, 360],
    tickvals=[0, 60, 120, 180, 240, 300, 360],
    ticktext=["0 red", "60 yellow", "120 green", "180 cyan", "240 blue", "300 magenta", "360 red"],
    title_text="Hue (degrees)",
    row=2,
    col=1,
)
fig.update_yaxes(title_text="Share of coloured pixels (%)", row=1, col=1)
fig.update_yaxes(visible=False, row=2, col=1)
fig.update_layout(title="Hue distribution of the coloured pixels per class", legend_title_text="class", height=500)
fig.show()

> So what we are clearly seeing here is that sunflowers do have a lot of yellow in their photos. 
> Yellow is a lot more rare for tulips and roses but they do occur for both of them in a similiar amount. > There's also a peak for cyan-blueish in sunflowers and a little for roses and tulips. Those are probably sky-colours!

> Green occurs for all of them in a similiar amount. 

> Roses and Tulips do have a lot of magenta and red values which makes sense

> So to sum this up: I would think that a model would very easily be able to distinguish sunflowers from tulips and roses but have a harder time between them since they have very similiar colour values. 
> What differences are there between roses and tulips in the photos?

In [ ]:
# one row per photo: its dominant hue (peak of the hue histogram) and the mean saturation and brightness
photo_rows = []
for folder in class_folders:
    for path in sorted(folder.glob("*.jpg")):
        rgb = keras.utils.load_img(path, target_size=IMAGE_SIZE)
        hsv = rgb_to_hsv(np.asarray(rgb) / 255).reshape(-1, 3)
        # same filter as before: only clearly coloured pixels
        colourful = hsv[(hsv[:, 1] > 0.3) & (hsv[:, 2] > 0.2) & (hsv[:, 2] < 0.95)]
        counts, _ = np.histogram(colourful[:, 0] * 360, bins=bins)
        photo_rows.append(
            {
                "class": folder.name,
                "file": path.name,
                "dominant_hue": centers[counts.argmax()],
                "mean_saturation": colourful[:, 1].mean(),
                "mean_value": colourful[:, 2].mean(),
            }
        )

photo_colours = pd.DataFrame(photo_rows)

colour_order = ["red", "orange", "yellow", "green", "blue", "purple/magenta"]
photo_colours["dominant_colour"] = photo_colours["dominant_hue"].map(colour_name)

# how many photos of each class have which dominant colour?
px.histogram(
    photo_colours,
    x="dominant_colour",
    color="class",
    barmode="group",
    histnorm="percent",
    category_orders={"dominant_colour": colour_order},
    title="Dominant colour per photo",
    labels={"dominant_colour": "Dominant colour of the photo", "percent": "Share of the photos of the class (%)"},
).show()

> In the first plot we were looking at all the colours in the photo. A very big flower in one colour would overpower the plot a little. 
> Here we are looking at only the dominant colour per photo to see whether there are any differences in roses and tulips. The only notable thing would probably be that Tulips have more photos with dominant green than roses in the training set. But other than that, they are very similiar again. 

In [ ]:
box_saturation = px.box(photo_colours, x="class", y="mean_saturation", color="class", points="all", hover_data=["file"])
box_value = px.box(photo_colours, x="class", y="mean_value", color="class", points="all", hover_data=["file"])

fig = make_subplots(rows=1, cols=2, subplot_titles=("Mean saturation", "Mean brightness"))
for trace in box_saturation.data:
    fig.add_trace(trace, row=1, col=1)
for trace in box_value.data:
    trace.showlegend = False  # the legend is the same as on the left
    fig.add_trace(trace, row=1, col=2)

fig.update_yaxes(title_text="Mean saturation (0-1)", row=1, col=1)
fig.update_yaxes(title_text="Mean brightness (0-1)", row=1, col=2)
fig.update_layout(width=1000, height=450, legend_title_text="class")
fig.show()

display(photo_colours.groupby("class")[["mean_saturation", "mean_value"]].agg(["mean", "std"]).round(2))

> The boxplots show us the saturation and brightness but tulips and roses are very similiar here again...
> I guess the model will have to learn from the shapes for roses and tulips since tulips are more closed that roses. 

## Repurpose a pretrained network

Use `MobileNetV2` from `keras.applications` as the pretrained network and repurpose it to classify the three flowers. Plot its learning curves: does it overfit?

> [!IMPORTANT]
> **Every image has to be prepared in exactly the same way**: the same `image_size`, the same `preprocess_input` of the base, and the same frozen base, for the training, the validation and later the test images.

In [ ]:
keras.utils.clear_session()
keras.utils.set_random_seed(42)
# a fresh training dataset in every training cell: its shuffling changes with every pass over the same dataset
# object, so reusing one object would give different batches when a cell is run again
train_ds = load_images("train", shuffle=True, data_dir=FLOWERS, image_size=IMAGE_SIZE)

# the pretrained network without its classifier, frozen: its weights are not changed by the training
base = mobilenet_v2.MobileNetV2(input_shape=IMAGE_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False

dense_head = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (3,)),
        # the model takes the raw photos (0-255) and scales them to -1..1 itself, which is what
        # mobilenet_v2.preprocess_input does. A Rescaling layer, unlike a Lambda layer, can be loaded again.
        keras.layers.Rescaling(1 / 127.5, offset=-1),
        base,
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(units=len(class_names), activation="softmax"),
    ]
)
dense_head.compile(loss="sparse_categorical_crossentropy", optimizer=keras.optimizers.Adam(1e-3), metrics=["accuracy"])
history = dense_head.fit(train_ds, validation_data=valid_ds, epochs=15, verbose=0)

curves = history_to_frame(history)
plot_learning_curves(curves, title="Dense head").show()

# one row per model; later models are added with pd.concat
n_correct, n_wrong = count_correct_and_wrong(dense_head, valid_ds)
results = summarize_curves(curves, "dense head", n_correct, n_wrong)
display(results)

Path("models").mkdir(exist_ok=True)
dense_head.save("models/dense_head.keras")

> This is overfitting quite badly, so we need to tweak it! Data augmentation will increase our amount of photos for the training as well as present it with a few different flowers. 

## If needed, improve the model with data augmentation

and compare the validation accuracy with step 4. Choose your best model by its validation accuracy. **Save every model to a file** with `model.save`, for example `models/dense_head.keras`, so that you can load all of them in step 6 with `keras.models.load_model`, even after restarting the notebook.

In [ ]:
# the layers get their own seed: they are created here, before the cell that sets the global seed, and
# without a seed their random flips, rotations and zooms would differ from run to run
augmentation = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal_and_vertical", seed=42),
        keras.layers.RandomRotation(0.20, seed=42),  # up to 5 % of a full turn in either direction
        keras.layers.RandomZoom(0.15, seed=42),
    ],
    name="augmentation",
)

images, labels = next(iter(train_ds))  # the first batch of 32 images
flower = images[:1]  # one image from the first training batch
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(flower[0].numpy().astype("uint8"))
axes[0].set_title("original")
for ax in axes[1:]:
    ax.imshow(augmentation(flower, training=True)[0].numpy().astype("uint8"))
    ax.set_title("augmented")
for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
keras.utils.clear_session()
keras.utils.set_random_seed(42)
train_ds = load_images("train", shuffle=True, data_dir=FLOWERS, image_size=IMAGE_SIZE)  # fresh, see the cell above

base = mobilenet_v2.MobileNetV2(input_shape=IMAGE_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False

# the same model as before, with the augmentation as its first layer. Keras only applies it while training,
# so the validation photos and the photos in the app stay unchanged.
# The augmentation is cloned: every use of a layer changes its random numbers, so the clone starts from its seed again.
dense_head_aug = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (3,)),
        keras.models.clone_model(augmentation),
        keras.layers.Rescaling(1 / 127.5, offset=-1),
        base,
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(units=len(class_names), activation="softmax"),
    ]
)
dense_head_aug.compile(loss="sparse_categorical_crossentropy", optimizer=keras.optimizers.Adam(1e-3), metrics=["accuracy"])
history_aug = dense_head_aug.fit(train_ds, validation_data=valid_ds, epochs=15, verbose=0)

curves_aug = history_to_frame(history_aug)
plot_learning_curves(curves_aug, title="Dense head with data augmentation").show()

n_correct, n_wrong = count_correct_and_wrong(dense_head_aug, valid_ds)
results = pd.concat(
    [results, summarize_curves(curves_aug, "dense head + augmentation", n_correct, n_wrong)], ignore_index=True
)
display(results)

dense_head_aug.save("models/dense_head_augmentation.keras")

## Tune the hyperparameters with Hyperband

Hyperband trains many models for only a few epochs, keeps the most promising third of them and trains those for longer, and repeats this until the best ones have trained for `max_epochs`. The base stays frozen and the photos are prepared exactly as before; the tuner searches the strength of the augmentation, the dropout, an optional hidden layer, the L2 penalty and the learning rate.

The objective is the validation **loss**: with only 60 validation photos, one photo changes the accuracy by 1.7 percentage points, so the loss is the less noisy signal.

In [ ]:
keras.utils.clear_session()
keras.utils.set_random_seed(42)
train_ds = load_images("train", shuffle=True, data_dir=FLOWERS, image_size=IMAGE_SIZE)  # fresh, see above

tuner = kt.Hyperband(
    lambda hp: build_model(hp, IMAGE_SIZE, len(class_names)),
    objective="val_loss",
    max_epochs=20,
    factor=3,
    seed=42,
    directory="tuner",  # the results are saved here, so a restarted search continues where it stopped
    project_name="dense_head",
)
tuner.search(
    train_ds,
    validation_data=valid_ds,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=4)],
    verbose=0,
)

best_hp = tuner.get_best_hyperparameters(1)[0]
display(pd.Series(best_hp.values, name="best value"))

In [ ]:
keras.utils.clear_session()
keras.utils.set_random_seed(42)
train_ds = load_images("train", shuffle=True, data_dir=FLOWERS, image_size=IMAGE_SIZE)  # fresh, see above

# train a fresh model with the best hyperparameters; early stopping gives back the weights of the epoch
# with the lowest validation loss, which is the model we save
dense_head_tuned = build_model(best_hp, IMAGE_SIZE, len(class_names))
history_tuned = dense_head_tuned.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=50,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)],
    verbose=0,
)

curves_tuned = history_to_frame(history_tuned)
plot_learning_curves(curves_tuned, title="Dense head tuned with Hyperband").show()

# the table row is the best epoch, because that is the model after restoring its weights
best_epoch = curves_tuned["val_loss"].idxmin()
n_correct, n_wrong = count_correct_and_wrong(dense_head_tuned, valid_ds)
results = pd.concat(
    [results, summarize_curves(curves_tuned.iloc[: best_epoch + 1], "dense head tuned", n_correct, n_wrong)],
    ignore_index=True,
)
display(results)

dense_head_tuned.save("models/dense_head_tuned.keras")

## Stronger augmentation, chosen by hand

Hyperband's best trials used moderate rotation and zoom (0.10-0.15) and no contrast change, so I keep those and add augmentations that change the position, the lighting and what is visible, but not the colour of the flower: translation, brightness and random erasing. No hue changes, because they could turn a sunflower into a different-looking flower.

In [ ]:
# a separate augmentation just for this preview, so that drawing the photos here does not change the
# random numbers that the augmentation of the model uses in the training cell below
preview_augmentation = make_strong_augmentation()

images, labels = next(iter(train_ds))  # the first batch of 32 images
fig, axes = plt.subplots(len(class_names), 6, figsize=(16, 3 * len(class_names)))
for row, (class_index, class_name) in zip(axes, enumerate(class_names)):
    flower = images[labels.numpy() == class_index][:1]  # the first photo of this class in the batch
    row[0].imshow(flower[0].numpy().astype("uint8"))
    row[0].set_title(f"original ({class_name})")
    for ax in row[1:]:
        ax.imshow(preview_augmentation(flower, training=True)[0].numpy().astype("uint8"))
        ax.set_title("augmented")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
keras.utils.clear_session()
keras.utils.set_random_seed(42)
train_ds = load_images("train", shuffle=True, data_dir=FLOWERS, image_size=IMAGE_SIZE)  # fresh, see above

base = mobilenet_v2.MobileNetV2(input_shape=IMAGE_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False

dense_head_manual = keras.Sequential(
    [
        keras.Input(shape=IMAGE_SIZE + (3,)),
        make_strong_augmentation(),
        keras.layers.Rescaling(1 / 127.5, offset=-1),
        base,
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(units=len(class_names), activation="softmax"),
    ]
)

# the learning rate starts at 1e-3 and shrinks by 10 % every epoch (one epoch is len(train_ds) batches)
learning_rate = keras.optimizers.schedules.ExponentialDecay(
    1e-3, decay_steps=len(train_ds), decay_rate=0.9
)
dense_head_manual.compile(
    loss="sparse_categorical_crossentropy", optimizer=keras.optimizers.Adam(learning_rate), metrics=["accuracy"]
)
history_manual = dense_head_manual.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=60,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True)],
    verbose=0,
)

curves_manual = history_to_frame(history_manual)
plot_learning_curves(curves_manual, title="Dense head with strong augmentation").show()

# the table row is the best epoch, because that is the model after restoring its weights
best_epoch = curves_manual["val_loss"].idxmin()
print(f"Best epoch: {best_epoch + 1} of {len(curves_manual)}")
n_correct, n_wrong = count_correct_and_wrong(dense_head_manual, valid_ds)
results = pd.concat(
    [results, summarize_curves(curves_manual.iloc[: best_epoch + 1], "dense head + strong augmentation", n_correct, n_wrong)],
    ignore_index=True,
)
display(results)

dense_head_manual.save("models/dense_head_strong_augmentation.keras")

## Evaluate all models on the test set

The saved models are loaded again from `models/`, so that this part also works after restarting the notebook. The test photos are loaded exactly like the validation photos (same `image_size`, `shuffle=False`). The test set has only 30 photos, so **one photo changes the test accuracy by 3.3 percentage points**.

In [ ]:
model_files = {
    "dense head": "models/dense_head.keras",
    "dense head + augmentation": "models/dense_head_augmentation.keras",
    "dense head tuned": "models/dense_head_tuned.keras",
    "dense head + strong augmentation": "models/dense_head_strong_augmentation.keras",
}
saved_models = {name: keras.models.load_model(path) for name, path in model_files.items()}

# the training photos again, but in their order: train_ds is shuffled, so its labels would not line up with the
# predictions. The augmentation layers of the models only act while training, so they do not change the photos here.
train_eval_ds = load_images("train", shuffle=False, data_dir=FLOWERS, image_size=IMAGE_SIZE)

# the training, the validation and the test accuracy of every model in one table
rows = []
for name, model in saved_models.items():
    train_correct, train_wrong = count_correct_and_wrong(model, train_eval_ds)
    val_correct, val_wrong = count_correct_and_wrong(model, valid_ds)
    test_correct, test_wrong = count_correct_and_wrong(model, test_ds)
    rows.append(
        {
            "model": name,
            "train_accuracy": train_correct / (train_correct + train_wrong),
            "val_accuracy": val_correct / (val_correct + val_wrong),
            "test_accuracy": test_correct / (test_correct + test_wrong),
            "test_wrong": test_wrong,
        }
    )
test_results = pd.DataFrame(rows)
display(test_results)

In [ ]:
fig = make_subplots(rows=1, cols=3, shared_yaxes=True, subplot_titles=("Training", "Validation", "Test"))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd"]  # one colour per model, the same in all three plots

for col, column in enumerate(["train_accuracy", "val_accuracy", "test_accuracy"], start=1):
    fig.add_trace(
        go.Bar(
            x=test_results[column],
            y=test_results["model"],
            orientation="h",
            marker_color=colors,
            text=test_results[column].map("{:.2f}".format),
            textposition="outside",
            showlegend=False,
        ),
        row=1,
        col=col,
    )

fig.update_xaxes(title_text="Accuracy", range=[0, 1.1])
fig.update_yaxes(autorange="reversed")  # the first model on top
fig.update_layout(title="Accuracy of every model", width=1100, height=400, margin=dict(l=230))
fig.show()


## Best Model

In [ ]:
# the best model is chosen by its validation accuracy, not by its test accuracy
best_name = test_results.loc[test_results["val_accuracy"].idxmax(), "model"]
print("Best model by validation accuracy:", best_name)


### Mistakes on validation data

In [ ]:
wrong_val = plot_confusion_and_mistakes(saved_models[best_name], valid_ds, class_names, title=f"{best_name}, validation data")

### Mistakes on test data

In [ ]:
wrong_test = plot_confusion_and_mistakes(saved_models[best_name], test_ds, class_names, title=f"{best_name}, test data")